# inference-mode-step — ex2: diagnose missing inference_mode decorator on step

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `inference-mode-step`. Running the final beacon cell reports progress against the `PyTorch: Inference mode step` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Inference mode step` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`inference-mode-step`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "inference-mode-step"
DD_SUBTOPIC = "PyTorch: Inference mode step"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `@t.inference_mode()` on `step` — quick refresher

PyTorch optimizers mutate parameters IN PLACE. The naive form `theta -= self.lr * g` on a leaf with `requires_grad=True` outside any no-grad context raises:

> *RuntimeError: a leaf Variable that requires grad is being used in an in-place operation.*

The fix is to declare `step` as living outside autograd's bookkeeping. Two equivalent decorators:

```
@t.no_grad()           # disables grad tracking inside the call
@t.inference_mode()    # stricter newer version (also kills version counters)
```

ARENA's SGD/RMSprop/Adam impls all use `@t.inference_mode()` on `step`. That single decorator is what lets the body do `theta -= ...` and `buffer.copy_(...)` without autograd screaming. It is a hard requirement, not a stylistic choice.

### Exercise 2 — diagnose missing inference_mode decorator on step

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the `leaf Variable ... in-place` runtime error raised by an optimizer whose `step` lacks `@t.inference_mode()`, and fix it by adding the decorator without otherwise changing the body.
> Keywords: leaf-in-place-error, debug, inference-mode-missing
> ```

**KCs targeted:** `inference-mode-decorator-wraps-step`, `leaf-in-place-error-signature`

Below is `BrokenSGD` — a hand-rolled SGD whose `step` body uses the bare in-place form `p -= self.lr * p.grad` but FORGETS to decorate `step`. The first `.step()` call raises a RuntimeError.

Your job: implement `Ex2FixedSGD` — identical to `BrokenSGD` EXCEPT that `step` is correctly decorated with `@t.inference_mode()`. Do NOT change the body of `step`. Do NOT switch to `p.data -= ...`. The fix is exactly one decorator.

Also implement `ex2_demonstrate_broken_raises(broken_opt)` — call `broken_opt.step()` inside a `try / except RuntimeError` and return the string of the exception (so the test can verify the characteristic error message).

In [ ]:
class BrokenSGD:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr

    def step(self):
        for p in self.params:
            if p.grad is not None:
                p -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = None


class Ex2FixedSGD:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr

    @t.inference_mode()       # <-- the entire fix
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = None


def ex2_demonstrate_broken_raises(broken_opt) -> str:
    try:
        broken_opt.step()
    except RuntimeError as e:
        return str(e)
    return ''


<details><summary>Solution</summary>

```python
class BrokenSGD:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr

    def step(self):
        for p in self.params:
            if p.grad is not None:
                p -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = None


class Ex2FixedSGD:
    def __init__(self, params, lr):
        self.params = list(params)
        self.lr = lr

    @t.inference_mode()       # <-- the entire fix
    def step(self):
        for p in self.params:
            if p.grad is not None:
                p -= self.lr * p.grad

    def zero_grad(self):
        for p in self.params:
            p.grad = None


def ex2_demonstrate_broken_raises(broken_opt) -> str:
    try:
        broken_opt.step()
    except RuntimeError as e:
        return str(e)
    return ''
```

**Diagnosis recipe.** When you see `a leaf Variable that requires grad is being used in an in-place operation` from an optimizer step, the cause is ALMOST ALWAYS one of:

1. Missing `@t.inference_mode()` / `@t.no_grad()` on `step`.
2. Using bare `p -= ...` outside such a block, in user code (e.g. a manual update inside a Jupyter cell).
3. Forgetting `.data` in a fix attempt that did NOT add the decorator.

The cleanest fix is the decorator — `.data` is older and PyTorch discourages it in new code. ARENA's reference SGD/Adam/RMSprop all use the decorator.

**Why the bare op raises at all.** Autograd tracks every operation that participates in a parameter's history. An in-place mutation invalidates that history (the input tensor no longer holds the values backward needs). For a NON-leaf tensor PyTorch detects this via version counters and raises at backward time; for a leaf it refuses up-front.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()